# Plotting VisProbe results

VisProbe deliberately does not bundle plotting helpers in the library -- matplotlib version drift and figure-style preferences belong in user code. This notebook is a copy-paste starting point. Edit it to taste.

In [ ]:
import matplotlib.pyplot as plt

from visprobe import CompositionalResults

results = CompositionalResults.load("./results")

## Grid: one subplot per scenario, one line per model

In [ ]:
def plot_compositional(results, save_path=None, figsize=(15, 10)):
    models = results.get_models()
    scenarios = results.get_scenarios()
    if not models or not scenarios:
        print("No data to plot")
        return

    n_scenarios = len(scenarios)
    n_cols = min(3, n_scenarios)
    n_rows = (n_scenarios + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize)
    if n_rows == 1 and n_cols == 1:
        axes = [[axes]]
    elif n_rows == 1:
        axes = [axes]
    elif n_cols == 1:
        axes = [[ax] for ax in axes]

    for idx, scenario in enumerate(scenarios):
        row, col = idx // n_cols, idx % n_cols
        ax = axes[row][col]
        for model_name in models:
            if scenario in results.data[model_name]:
                severities = sorted(results.data[model_name][scenario].keys())
                accs = [results.data[model_name][scenario][s].accuracy for s in severities]
                ax.plot(severities, accs, marker="o", label=model_name)
        ax.set_xlabel("Severity")
        ax.set_ylabel("Accuracy")
        ax.set_title(scenario)
        ax.legend()
        ax.grid(True, alpha=0.3)
        ax.set_ylim([0, 1])

    for idx in range(n_scenarios, n_rows * n_cols):
        row, col = idx // n_cols, idx % n_cols
        axes[row][col].axis("off")

    plt.suptitle("Compositional Robustness Results", fontsize=14)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    else:
        plt.show()


plot_compositional(results)

## Single scenario, accuracy vs severity

In [ ]:
scenario = "noise"
fig, ax = plt.subplots(figsize=(8, 5))
for model in results.get_models():
    severities = sorted(results.data[model][scenario].keys())
    accs = [results.data[model][scenario][s].accuracy for s in severities]
    ax.plot(severities, accs, marker="o", label=model)
ax.set_xlabel("Severity")
ax.set_ylabel("Accuracy")
ax.set_title(f"Accuracy vs severity ({scenario})")
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim([0, 1])
plt.show()